# micro:bit Level

In this demo, we deploy a small MicroPython program to the micro:bit and instruct it to return it's accelerometer data ever 0.05 seconds. Once this is running, in the next cell, we read in this data and to create a floating "level" in 3D space!

## Connect to the micro:bit and deploy

Attach your micro:bit via the serial cable. In the next cell, connect to the device and deploy.

The code runs in a `while True:` loop, where it reads the x and y values from the accelerometer and prints them to the serial output.

When the code is deployed, you can open the serial monitor tab to check the values being returned.

## Create our 3D objects

In the next code cell, we are going to create 3D objects for our scene. This includes a `level` object, which is a thin box, and two text objects (to display the X and Y values).

In [ ]:
from codetto import scene3d

level = scene3d.Shapes.Box(width=6, height=0.5, depth=6)
level.set_alpha(0.5)

x_text = scene3d.Shapes.Text("X")
x_text.set_position(-3,2,0)

y_text = scene3d.Shapes.Text("Y")
y_text.set_position(3,2,0)

print("3D objects created")

## Create our 3D scene and read the micro:bit data

In our final cell, we create the 3D scene and on every frame, read the data from the micro:bit and update the 3D objects in the scene.

There are a couple of important constants here:

- `MG_CONV` converts milli-gs (reported from the accelerometer) to degrees in the 3D scene.
- `SMOOTHING` is a smoothing factor. The accelerometer is very sensitive to movement, so this helps prevent shaking in the scene. Try setting this to 1 (no smoothing) and see what happens!

In [ ]:
from codetto import microbit, scene3d

scene = scene3d.Scene()
scene.set_sky(scene3d.Sky.CLOUDS)
scene.camera.set_position(0, 1, -10).look_at(0, 0, 0)

scene.add(level)
scene.add(x_text)
scene.add(y_text)

MG_CONV = 1000 / 90 # Convert milli-g to degrees

SMOOTHING = 0.15 # Smoothing factor
smooth_x = 0
smooth_y = 0

@scene.on_frame
def tick(dt):
  global smooth_x, smooth_y

  line = microbit.read_line_nowait()
  if line is not None:
    x, y = map(int, line.split(","))
    smooth_x += (x - smooth_x) * SMOOTHING
    smooth_y += (y - smooth_y) * SMOOTHING

    # Update the 3D scene
    level.set_rotation(0 - smooth_y / MG_CONV, 0, 0 - smooth_x / MG_CONV)
    x_text.set_text(f"X: {smooth_x / MG_CONV:.1f}")
    y_text.set_text(f"Y: {smooth_y / MG_CONV:.1f}")
      
if not microbit.is_connected():
    print("Connect via the device cell above first")
else:
    microbit.clear_buffer()
    scene.run()